# 画像をまとめてリサイズ・形式変換

Colabに画像をアップロードし、長辺の上限に合わせて縮小してから、WebP・JPEG・PNGのいずれかに一括変換します。SNS投稿用の軽量化や、複数画像の形式統一に使えます。

## 使い方
1. 上から順にセルを実行します。
2. 設定セルで出力形式・最大辺・画質を調整します。
3. 実行セルで画像を選ぶと、変換結果をZIPでダウンロードできます。

アップロードしたファイルはこのランタイム内だけで処理します。入力ファイルは変更せず、変換結果は別フォルダーに保存します。容量保護のため、画像の合計アップロード量は200 MBまで、画像1枚あたりの画素数は5,000万までです。アニメーション画像は誤った静止画化を避けるためスキップします。ランタイムを終了するとファイルは消去されます。

In [ ]:
%pip -q install Pillow

from pathlib import Path
from PIL import Image, ImageOps, UnidentifiedImageError
from google.colab import files
import zipfile
import tempfile

print('準備できました。次の設定セルを確認してください。')

## 変換設定
出力形式は `webp`、`jpeg`、`png` から選びます。`MAX_SIDE = None` にするとサイズを変更しません。JPEGは透明部分を白で塗りつぶします。

In [ ]:
OUTPUT_FORMAT = 'webp'  # 'webp' / 'jpeg' / 'png'
MAX_SIDE = 1920          # 長辺の最大ピクセル数。None ならリサイズしない
QUALITY = 85             # WebP/JPEG の画質。1〜95

if OUTPUT_FORMAT not in {'webp', 'jpeg', 'png'}:
    raise ValueError("OUTPUT_FORMAT は 'webp'、'jpeg'、'png' のいずれかにしてください。")
if MAX_SIDE is not None and (not isinstance(MAX_SIDE, int) or MAX_SIDE < 1):
    raise ValueError('MAX_SIDE は正の整数または None にしてください。')
if not isinstance(QUALITY, int) or not 1 <= QUALITY <= 95:
    raise ValueError('QUALITY は 1〜95 の整数にしてください。')

print(f'形式: {OUTPUT_FORMAT} / 最大辺: {MAX_SIDE or "変更なし"} / 画質: {QUALITY}')

## アップロードして一括変換
対応形式はJPEG・PNG・WebP・BMP・TIFFです。ZIP内のファイル名は元の名前を引き継ぎ、同名ファイルがあれば番号を付けます。

In [ ]:
uploaded = files.upload()
if not uploaded:
    raise RuntimeError('画像が選択されませんでした。セルを再実行してください。')

MAX_TOTAL_UPLOAD_BYTES = 200 * 1024 * 1024
MAX_IMAGE_PIXELS = 50_000_000
total_upload_bytes = sum(len(data) for data in uploaded.values())
if total_upload_bytes > MAX_TOTAL_UPLOAD_BYTES:
    raise ValueError(f'アップロード合計が上限の200 MBを超えています（{total_upload_bytes / 1024**2:.1f} MB）。ファイル数を分けて再実行してください。')

# 以前の実行結果やランタイム上のファイルを消さないよう、実行ごとに別フォルダーを使う
work_dir = Path(tempfile.mkdtemp(prefix='image_conversion_', dir='/content'))
input_dir = work_dir / 'input'
output_dir = work_dir / 'converted'
input_dir.mkdir(parents=True)
output_dir.mkdir()

# 同名ファイルを安全に扱うため、アップロード内容を個別の一時ファイルとして保存
saved_inputs = []
for index, (name, data) in enumerate(uploaded.items(), start=1):
    safe_name = Path(name).name
    if not safe_name:
        safe_name = f'image_{index}'
    src = input_dir / f'{index:04d}_{safe_name}'
    src.write_bytes(data)
    saved_inputs.append((safe_name, src))

allowed_formats = {'JPEG', 'PNG', 'WEBP', 'BMP', 'TIFF'}
extension = {'webp': '.webp', 'jpeg': '.jpg', 'png': '.png'}[OUTPUT_FORMAT]
seen_names = set()
converted = []
skipped = []

for original_name, src in saved_inputs:
    try:
        with Image.open(src) as opened:
            if opened.format not in allowed_formats:
                skipped.append((original_name, f'未対応形式: {opened.format}'))
                continue
            if getattr(opened, 'n_frames', 1) > 1:
                skipped.append((original_name, 'アニメーション画像'))
                continue
            width, height = opened.size
            if width * height > MAX_IMAGE_PIXELS:
                skipped.append((original_name, f'画素数が上限の5,000万を超過 ({width}×{height})'))
                continue
            image = ImageOps.exif_transpose(opened)
            image.load()

        if MAX_SIDE is not None and max(image.size) > MAX_SIDE:
            image.thumbnail((MAX_SIDE, MAX_SIDE), Image.Resampling.LANCZOS)

        if OUTPUT_FORMAT == 'jpeg':
            if image.mode in ('RGBA', 'LA') or (image.mode == 'P' and 'transparency' in image.info):
                rgba = image.convert('RGBA')
                background = Image.new('RGB', rgba.size, 'white')
                background.paste(rgba, mask=rgba.getchannel('A'))
                image = background
            else:
                image = image.convert('RGB')
        elif OUTPUT_FORMAT == 'png':
            if image.mode not in ('RGB', 'RGBA', 'L', 'LA', 'P'):
                image = image.convert('RGBA' if ('A' in image.getbands() or 'transparency' in image.info) else 'RGB')
        else:  # WebP
            if image.mode not in ('RGB', 'RGBA'):
                image = image.convert('RGBA' if 'A' in image.getbands() else 'RGB')

        base = Path(original_name).stem or 'image'
        output_name = f'{base}{extension}'
        counter = 2
        while output_name.lower() in seen_names:
            output_name = f'{base}_{counter}{extension}'
            counter += 1
        seen_names.add(output_name.lower())
        destination = output_dir / output_name

        save_options = {}
        if OUTPUT_FORMAT in {'webp', 'jpeg'}:
            save_options = {'quality': QUALITY, 'optimize': True}
        if OUTPUT_FORMAT == 'webp':
            save_options['method'] = 6
        elif OUTPUT_FORMAT == 'png':
            save_options = {'optimize': True}
        image.save(destination, format={'webp': 'WEBP', 'jpeg': 'JPEG', 'png': 'PNG'}[OUTPUT_FORMAT], **save_options)
        converted.append((original_name, output_name, src.stat().st_size, destination.stat().st_size, image.size))
    except (UnidentifiedImageError, OSError, Image.DecompressionBombError) as exc:
        skipped.append((original_name, f'読み込みに失敗: {exc}'))

print(f'変換: {len(converted)} 件 / スキップ: {len(skipped)} 件')
for src_name, dst_name, src_bytes, dst_bytes, size in converted:
    change = (1 - dst_bytes / src_bytes) * 100 if src_bytes else 0
    print(f'{src_name} → {dst_name}  |  {size[0]}×{size[1]} px  |  {src_bytes/1024:.0f} KB → {dst_bytes/1024:.0f} KB ({change:+.1f}%)')
for name, reason in skipped:
    print(f'スキップ: {name} — {reason}')

if not converted:
    raise RuntimeError('変換できる画像がありませんでした。対応形式とファイル内容を確認してください。')

zip_path = work_dir / 'converted_images.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(output_dir.iterdir()):
        archive.write(path, arcname=path.name)
files.download(str(zip_path))